# LeetCode 1897 — Redistribute Characters to Make All Strings Equal bằng Python

Link đề: https://leetcode.com/problems/redistribute-characters-to-make-all-strings-equal/

Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

```python
class Solution:
    def makeEqual(self, words: list[str]) -> bool:

```

**Mục tiêu học:**

1. Đọc hiểu một bài "có phép biến đổi": phép toán cho phép làm gì, và **cái gì không đổi** (bất biến) sau mỗi phép.
2. Nhận ra: ký tự di chuyển tự do → **chỉ tần suất** (số lần xuất hiện) của mỗi ký tự là quan trọng.
3. Thành thạo 3 công cụ đếm: `dict` + `get`, `collections.Counter`, mảng 26 phần tử với `ord`.
4. Chứng minh điều kiện **cần và đủ**: mỗi ký tự xuất hiện số lần chia hết cho `n = len(words)`.
5. Tránh bẫy "chỉ kiểm tra tổng độ dài chia hết cho n".

## 1. Bài toán yêu cầu gì?

Cho mảng chuỗi `words`. Một **phép toán**: chọn hai chỉ số khác nhau `i != j`, với `words[i]` khác rỗng, lấy **một ký tự bất kỳ** của `words[i]` và chèn vào **vị trí bất kỳ** của `words[j]`.

Hỏi: sau một số phép toán tùy ý, có thể làm **mọi** chuỗi bằng nhau không?

| words | Kết quả | Vì sao |
|---|:---:|---|
| `["abc","aabc","bc"]` | `True` | chuyển một `'a'` từ `"aabc"` sang đầu `"bc"` → `["abc","abc","abc"]` |
| `["ab","a"]` | `False` | có 2 chữ `a` nhưng chỉ 1 chữ `b` — không chia đều `b` cho 2 chuỗi |
| `["a"]` | `True` | chỉ một chuỗi thì luôn "bằng nhau" |

**Ràng buộc:** `1 <= len(words) <= 100`, `1 <= len(words[i]) <= 100`, chỉ gồm chữ thường.

In [1]:
examples = [
    (["abc", "aabc", "bc"], True),
    (["ab", "a"], False),
    (["a"], True),
]
for words, expected in examples:
    print(f"{words} -> kỳ vọng {expected}")

['abc', 'aabc', 'bc'] -> kỳ vọng True
['ab', 'a'] -> kỳ vọng False
['a'] -> kỳ vọng True


## 2. Nền tảng Python cần biết

### 2.1. `"".join(words)` — gộp tất cả chuỗi

Vì ký tự di chuyển tự do giữa các chuỗi, ta thường quan tâm **toàn bộ** ký tự. `"".join(list_chuoi)` ghép chúng thành một chuỗi dài.

In [2]:
words = ["abc", "aabc", "bc"]
all_chars = "".join(words)
print(all_chars, len(all_chars))

abcaabcbc 9


### 2.2. Đếm bằng `dict` và `dict.get(key, default)`

- `cnt = {}` là dict rỗng.
- `cnt.get(ch, 0)` trả về `cnt[ch]` nếu có, **ngược lại 0** — không bị `KeyError`.
- Mẫu đếm kinh điển: `cnt[ch] = cnt.get(ch, 0) + 1`.

In [3]:
cnt = {}
for ch in "aabc":
    cnt[ch] = cnt.get(ch, 0) + 1
    print(f"gặp {ch!r} -> {cnt}")

try:
    print(cnt["z"])
except KeyError as e:
    print("Truy cập key không tồn tại -> KeyError:", e)
print("Dùng get:", cnt.get("z", 0))

gặp 'a' -> {'a': 1}
gặp 'a' -> {'a': 2}
gặp 'b' -> {'a': 2, 'b': 1}
gặp 'c' -> {'a': 2, 'b': 1, 'c': 1}
Truy cập key không tồn tại -> KeyError: 'z'
Dùng get: 0


### 2.3. `collections.Counter` — dict chuyên để đếm

`Counter(iterable)` đếm sẵn cho ta. Truy cập key không có trả về `0` (không lỗi). `.values()` cho các số đếm; `.most_common()` sắp theo tần suất.

In [4]:
from collections import Counter

counter = Counter("abcaabcbc")
print(counter)
print(counter["a"], counter["z"])        # 3, 0
print(list(counter.values()))
print(counter.most_common(2))

Counter({'a': 3, 'b': 3, 'c': 3})
3 0
[3, 3, 3]
[('a', 3), ('b', 3)]


### 2.4. Mảng 26 phần tử với `ord(ch) - ord('a')`

Chỉ có 26 chữ thường → dùng list độ dài 26, ô `k` đếm chữ cái thứ `k`. `ord('a') = 97`, nên `ord(ch) - 97` cho `0..25`.

In [5]:
freq = [0] * 26
for ch in "abcaabcbc":
    freq[ord(ch) - ord("a")] += 1
print(freq[:5])     # a, b, c, d, e
print({chr(97 + k): v for k, v in enumerate(freq) if v})

[3, 3, 3, 0, 0]
{'a': 3, 'b': 3, 'c': 3}


### 2.5. Phép `%` (chia lấy dư) để kiểm tra chia hết

`a % n == 0` nghĩa là `a` chia hết cho `n`. Ví dụ `6 % 3 == 0`, `7 % 3 == 1`. Và `0 % n == 0` — số 0 luôn chia hết.

In [6]:
for a in [0, 3, 6, 7, 1]:
    print(f"{a} % 3 = {a % 3} -> chia hết? {a % 3 == 0}")

0 % 3 = 0 -> chia hết? True
3 % 3 = 0 -> chia hết? True
6 % 3 = 0 -> chia hết? True
7 % 3 = 1 -> chia hết? False
1 % 3 = 1 -> chia hết? False


## 3. Tính bằng tay và tìm "cái không đổi"

Hãy tự hỏi: **một phép toán làm thay đổi gì, và giữ nguyên gì?**

- Thay đổi: độ dài từng chuỗi, thứ tự ký tự.
- **Giữ nguyên:** tổng số lần xuất hiện của **mỗi chữ cái** trên toàn mảng (ta chỉ di chuyển, không tạo/xóa ký tự).

Nếu cuối cùng `n` chuỗi đều bằng một chuỗi `S`, thì chữ `'a'` xuất hiện `n × (số 'a' trong S)` lần tổng cộng → phải **chia hết cho n**. Tương tự cho mọi chữ.

Ví dụ 1: tổng `a:3, b:3, c:3`, `n = 3` → mỗi chuỗi nhận `a:1, b:1, c:1` → `S = "abc"`.

Ví dụ 2: tổng `a:2, b:1`, `n = 2` → `b:1` không chia đôi được → `False`.

In [7]:
for words, expected in examples:
    n = len(words)
    total = Counter("".join(words))
    share = {ch: (v, v % n == 0) for ch, v in total.items()}
    print(f"{words}: n={n}, (tổng, chia hết?) = {share} -> kỳ vọng {expected}")

['abc', 'aabc', 'bc']: n=3, (tổng, chia hết?) = {'a': (3, True), 'b': (3, True), 'c': (3, True)} -> kỳ vọng True
['ab', 'a']: n=2, (tổng, chia hết?) = {'a': (2, True), 'b': (1, False)} -> kỳ vọng False
['a']: n=1, (tổng, chia hết?) = {'a': (1, True)} -> kỳ vọng True


## 4. Cách nghĩ "ngây thơ" và phản ví dụ

Nhiều bạn nghĩ: "Muốn chuỗi bằng nhau thì độ dài phải bằng nhau → chỉ cần **tổng độ dài chia hết cho n**".

Điều này là **cần** nhưng **không đủ**: hai chuỗi bằng nhau thì không chỉ cùng độ dài mà còn cùng **từng chữ cái**.

In [8]:
def make_equal_length_only(words):
    return sum(len(w) for w in words) % len(words) == 0

counter_example = ["ab", "aa"]   # tổng độ dài 4, chia hết cho 2
print("Chỉ xét độ dài:", make_equal_length_only(counter_example))
print("Đếm chữ:", Counter("".join(counter_example)))
print("-> a:3, b:1 không chia đôi được, đáp án đúng là False")

Chỉ xét độ dài: True
Đếm chữ: Counter({'a': 3, 'b': 1})
-> a:3, b:1 không chia đôi được, đáp án đúng là False


## 5. Cách 1 — Duyệt từng chữ cái khác nhau, dùng `str.count`

Baseline dễ viết: với mỗi chữ cái **khác nhau** trong toàn bộ ký tự, đếm số lần xuất hiện bằng `all_chars.count(ch)` rồi kiểm tra chia hết.

`set(all_chars)` cho các chữ cái khác nhau (không trùng).

In [9]:
def make_equal_count(words):
    n = len(words)
    all_chars = "".join(words)
    for ch in set(all_chars):
        if all_chars.count(ch) % n != 0:
            return False
    return True

for words, expected in examples:
    print(make_equal_count(words), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


**Độ phức tạp Cách 1:** gọi `L` = tổng số ký tự (≤ 100 × 100 = 10.000), `σ` = số chữ khác nhau (≤ 26). Mỗi `count` quét cả `L` ký tự → `O(σ · L)` ≈ 26 × 10.000. Vẫn nhanh, nhưng ta **quét lại** chuỗi tới 26 lần.

**Nút thắt:** đếm được trong **một** lượt duyệt — mỗi ký tự cộng 1 vào ô của nó.

## 6. Lời giải chính — Đếm một lượt bằng `Counter`

**Đoạn 1:** gộp và đếm một lần: `total = Counter("".join(words))`.

**Đoạn 2:** `n = len(words)`.

**Đoạn 3:** mọi số đếm chia hết cho `n`: `all(v % n == 0 for v in total.values())`.

In [10]:
def make_equal_counter(words):
    n = len(words)
    total = Counter("".join(words))
    return all(v % n == 0 for v in total.values())

for words, expected in examples:
    print(make_equal_counter(words), "| kỳ vọng", expected)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True


Không cần `"".join`: có thể cập nhật `Counter` theo từng chuỗi bằng `total.update(word)`.

In [11]:
total = Counter()
for w in ["abc", "aabc", "bc"]:
    total.update(w)
    print(f"sau {w!r:7}: {dict(total)}")

sau 'abc'  : {'a': 1, 'b': 1, 'c': 1}
sau 'aabc' : {'a': 3, 'b': 2, 'c': 2}
sau 'bc'   : {'a': 3, 'b': 3, 'c': 3}


## 7. Cách 3 và 4 — Tự đếm bằng `dict` và bằng mảng 26

Hai cách này không cần import, cho thấy rõ bên trong `Counter` làm gì. Mảng 26 còn dùng được ở mọi ngôn ngữ (C++, Java).

In [12]:
def make_equal_dict(words):
    n = len(words)
    cnt = {}
    for w in words:
        for ch in w:
            cnt[ch] = cnt.get(ch, 0) + 1
    for v in cnt.values():
        if v % n != 0:
            return False
    return True


def make_equal_array(words):
    n = len(words)
    freq = [0] * 26
    for w in words:
        for ch in w:
            freq[ord(ch) - ord("a")] += 1
    return all(f % n == 0 for f in freq)     # ô bằng 0 vẫn chia hết


for words, expected in examples:
    print(make_equal_dict(words), make_equal_array(words), "| kỳ vọng", expected)

True True | kỳ vọng True
False False | kỳ vọng False
True True | kỳ vọng True


Chú ý ở `make_equal_array`: các chữ không xuất hiện có `freq = 0`, và `0 % n == 0` → không làm hỏng kết quả. Đó là lý do có thể kiểm tra cả 26 ô mà không cần lọc.

## 8. TRACE từng bước

Ví dụ `words = ["abc", "aabc", "bc"]` (n = 3):

| Chuỗi đang đọc | Ký tự | `cnt` sau khi cộng |
|---|---|---|
| `"abc"` | a, b, c | `{a:1, b:1, c:1}` |
| `"aabc"` | a, a, b, c | `{a:3, b:2, c:2}` |
| `"bc"` | b, c | `{a:3, b:3, c:3}` |

Kiểm tra: `3 % 3 = 0` cho cả ba → `True`.

Hàm trace in từng ký tự và quyết định cuối; thử thêm ví dụ `False` để thấy chữ "phá" kết quả.

In [13]:
def trace_make_equal(words):
    n = len(words)
    cnt = {}
    print(f"n = {n}")
    for idx, w in enumerate(words):
        for ch in w:
            cnt[ch] = cnt.get(ch, 0) + 1
        print(f"  sau words[{idx}]={w!r:7}: cnt = {cnt}")
    for ch in sorted(cnt):
        v = cnt[ch]
        verdict = "chia hết" if v % n == 0 else "KHÔNG chia hết -> False"
        print(f"  {ch!r}: {v} % {n} = {v % n} -> {verdict}")
        if v % n != 0:
            return False
    print("  Mọi chữ đều chia hết -> True")
    return True

trace_make_equal(["abc", "aabc", "bc"])
print()
trace_make_equal(["ab", "a"])

n = 3
  sau words[0]='abc'  : cnt = {'a': 1, 'b': 1, 'c': 1}
  sau words[1]='aabc' : cnt = {'a': 3, 'b': 2, 'c': 2}
  sau words[2]='bc'   : cnt = {'a': 3, 'b': 3, 'c': 3}
  'a': 3 % 3 = 0 -> chia hết
  'b': 3 % 3 = 0 -> chia hết
  'c': 3 % 3 = 0 -> chia hết
  Mọi chữ đều chia hết -> True

n = 2
  sau words[0]='ab'   : cnt = {'a': 1, 'b': 1}
  sau words[1]='a'    : cnt = {'a': 2, 'b': 1}
  'a': 2 % 2 = 0 -> chia hết
  'b': 1 % 2 = 1 -> KHÔNG chia hết -> False


False

### Trace ca phản ví dụ "độ dài chia hết nhưng chữ không chia hết"

In [14]:
trace_make_equal(["ab", "aa"])

n = 2
  sau words[0]='ab'   : cnt = {'a': 1, 'b': 1}
  sau words[1]='aa'   : cnt = {'a': 3, 'b': 1}
  'a': 3 % 2 = 1 -> KHÔNG chia hết -> False


False

## 9. Code nộp LeetCode

In [15]:
from collections import Counter


class Solution:
    def makeEqual(self, words: list[str]) -> bool:
        n = len(words)
        total = Counter("".join(words))
        return all(v % n == 0 for v in total.values())


sol = Solution()
for words, expected in examples:
    assert sol.makeEqual(words) == expected
print("Solution (Counter) PASS")

Solution (Counter) PASS


Bản không import (mảng 26) — cũng Accepted:

In [16]:
class SolutionArray:
    def makeEqual(self, words: list[str]) -> bool:
        n = len(words)
        freq = [0] * 26
        for w in words:
            for ch in w:
                freq[ord(ch) - 97] += 1
        return all(f % n == 0 for f in freq)


for words, expected in examples:
    assert SolutionArray().makeEqual(words) == expected
print("SolutionArray PASS")

SolutionArray PASS


## 10. Vì sao đúng? (cần và đủ)

Gọi `total[c]` là tổng số lần chữ `c` xuất hiện, `n = len(words)`.

**Bất biến:** mỗi phép toán chỉ **di chuyển** một ký tự, nên `total[c]` không đổi với mọi `c`.

**Điều kiện cần.** Nếu cuối cùng mọi chuỗi bằng `S`, thì `total[c] = n × S.count(c)` → `total[c] % n == 0`.

**Điều kiện đủ.** Giả sử mọi `total[c] % n == 0`. Đặt `S` = chuỗi chứa mỗi chữ `c` đúng `total[c] / n` lần. Ta xây được bằng các phép toán:

- Với mỗi chuỗi đang **thừa** một chữ `c` (so với `S`), chuyển chữ đó sang một chuỗi đang **thiếu** `c`. Vì tổng thừa = tổng thiếu (cùng bằng `total[c] − n × S.count(c) = 0` khi cộng lại), luôn có chỗ nhận.
- Phép toán cho chèn **vị trí bất kỳ**, nên thứ tự ký tự cũng sắp được giống `S`.

Lưu ý nhỏ: phép toán yêu cầu `words[i]` khác rỗng — luôn thỏa vì ta chỉ lấy từ chuỗi đang **thừa** ký tự. Với `n = 1`, mọi số chia hết cho 1 → `True`, đúng vì một chuỗi luôn bằng chính nó.

## 11. Độ phức tạp và bảng so sánh

`L` = tổng số ký tự của mọi chuỗi, `σ` = số chữ cái khác nhau (≤ 26).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|:---:|:---:|---|
| Ngây thơ (chỉ độ dài) | tổng độ dài % n | `O(n)` | `O(1)` | **SAI** — chỉ là điều kiện cần |
| 1. `str.count` | đếm lại cho từng chữ | `O(σ·L)` | `O(L)` (chuỗi gộp) | dễ viết, quét nhiều lần |
| 2. `Counter` | đếm một lượt | `O(L)` | `O(σ)` + `O(L)` nếu join | ngắn, Pythonic |
| 3. `dict` thủ công | `get(ch, 0) + 1` | `O(L)` | `O(σ)` | hiểu bản chất Counter |
| 4. Mảng 26 | `freq[ord(ch)-97]` | `O(L)` | `O(26) = O(1)` | dùng được mọi ngôn ngữ |

## 12. Bộ test kiểm tra

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:34} PASS")

solvers = [make_equal_count, make_equal_counter, make_equal_dict, make_equal_array]

tests = [
    ("ví dụ 1", ["abc", "aabc", "bc"], True),
    ("ví dụ 2", ["ab", "a"], False),
    ("một chuỗi", ["xyz"], True),
    ("độ dài chia hết nhưng sai", ["ab", "aa"], False),
    ("đã bằng nhau", ["ab", "ba", "ab"], True),
    ("một chuỗi mang hết", ["aaaabbbb", "a", "b", "ab"], False),
    ("chia đều 4", ["aaaa", "bbbb", "cccc", "abc"], False),
    ("chia đều 2", ["aabb", "ab", "ab"], False),
    ("cùng chữ", ["a", "a", "a"], True),
    ("tổng chữ chia hết", ["zz", "z", "zzz"], True),
]
for name, w, e in tests:
    for f in solvers:
        assert f(w) == e, (name, f.__name__)
    check(name + " (4 cách)", make_equal_counter(w), e)

ví dụ 1 (4 cách)                   PASS
ví dụ 2 (4 cách)                   PASS
một chuỗi (4 cách)                 PASS
độ dài chia hết nhưng sai (4 cách) PASS
đã bằng nhau (4 cách)              PASS
một chuỗi mang hết (4 cách)        PASS
chia đều 4 (4 cách)                PASS
chia đều 2 (4 cách)                PASS
cùng chữ (4 cách)                  PASS
tổng chữ chia hết (4 cách)         PASS


Giải thích vài ca: `["aaaabbbb","a","b","ab"]`: `n=4`, `a:6, b:6` → 6 % 4 ≠ 0 → `False`. `["zz","z","zzz"]`: `z:6`, `n=3` → `True` (mỗi chuỗi thành `"zz"`).

Kiểm thử đối chiếu với một **mô phỏng tham lam** độc lập: thực sự "di chuyển" ký tự giữa các chuỗi để thử dựng `S` (oracle chậm nhưng bám sát định nghĩa phép toán).

In [18]:
import random

def simulate_redistribute(words):
    """Oracle: tham lam chuyển từng ký tự thừa sang chuỗi thiếu, rồi xem có bằng nhau không."""
    n = len(words)
    total = Counter("".join(words))
    if any(v % n for v in total.values()):
        return False          # bất biến: không thể dựng được S
    S = {ch: v // n for ch, v in total.items()}
    cur = [Counter(w) for w in words]
    moves = 0
    for ch in S:
        givers = [i for i in range(n) if cur[i][ch] > S[ch]]
        takers = [i for i in range(n) if cur[i][ch] < S[ch]]
        while givers and takers:
            g, t = givers[-1], takers[-1]
            cur[g][ch] -= 1; cur[t][ch] += 1; moves += 1
            if cur[g][ch] == S[ch]: givers.pop()
            if cur[t][ch] == S[ch]: takers.pop()
    return all(+cur[i] == +cur[0] for i in range(n))

random.seed(1897)
mismatch = 0
for _ in range(1500):
    k = random.randint(1, 5)
    ws = ["".join(random.choice("abc") for _ in range(random.randint(1, 6))) for _ in range(k)]
    expected = simulate_redistribute(ws)
    if {f(ws) for f in solvers} != {expected}:
        mismatch += 1
print("Số ca lệch so với mô phỏng:", mismatch)
assert mismatch == 0

Số ca lệch so với mô phỏng: 0


Trong oracle, `+counter` là cú pháp loại bỏ các key có giá trị ≤ 0, để hai `Counter` so sánh công bằng.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra tổng độ dài

Đã thấy ở mục 4: `["ab","aa"]` có tổng độ dài 4 chia hết cho 2 nhưng đáp án `False`.

### Lỗi 2: Đếm theo từng chuỗi thay vì toàn mảng

```python
for w in words:
    if Counter(w) != Counter(words[0]):   # SAI: bỏ qua việc được di chuyển ký tự
        return False
```

Đề cho phép chuyển ký tự giữa các chuỗi, nên ví dụ 1 sẽ bị trả nhầm `False`.

In [19]:
def wrong_per_word(words):
    return all(Counter(w) == Counter(words[0]) for w in words)

print("Sai :", wrong_per_word(["abc", "aabc", "bc"]), "| Đúng:", make_equal_counter(["abc", "aabc", "bc"]))

Sai : False | Đúng: True


### Lỗi 3: Chia cho số chữ cái khác nhau thay vì số chuỗi

```python
return all(v % len(total) == 0 for v in total.values())   # SAI: len(total) là số chữ khác nhau
```

`n` phải là `len(words)` — số chuỗi cần chia đều.

In [20]:
def wrong_divisor(words):
    total = Counter("".join(words))
    return all(v % len(total) == 0 for v in total.values())

w = ["abab", "ab", "ab"]   # a:4, b:4; có 2 chữ khác nhau nhưng 3 chuỗi
print("Sai :", wrong_divisor(w), "| Đúng:", make_equal_counter(w))

Sai : True | Đúng: False


### Lỗi 4: Quên `% n == 0` mà viết `% n` dạng truthy ngược chiều

```python
return all(v % n for v in total.values())   # SAI: v % n == 0 là falsy!
```

`v % n` bằng 0 (falsy) khi **chia hết**, nên biểu thức trên trả `True` khi **không** chia hết — ngược hoàn toàn. Viết rõ `== 0` để tránh nhầm.

In [21]:
def wrong_truthy(words):
    n = len(words)
    return all(v % n for v in Counter("".join(words)).values())

print("Sai :", wrong_truthy(["abc", "aabc", "bc"]), "| Đúng:", make_equal_counter(["abc", "aabc", "bc"]))

Sai : False | Đúng: True


## 14. Mẹo nhận dạng pattern và biến thể

- Phép toán cho **di chuyển / hoán đổi / sắp xếp lại tự do** ký tự → thứ tự không quan trọng, **chỉ tần suất** quan trọng → đếm bằng `Counter`/mảng 26.
- Câu hỏi "chia đều cho n phần" → kiểm tra `% n == 0` cho từng loại.
- Luôn tìm **bất biến** của phép toán: cái gì không đổi thì quyết định cái gì đạt được.

Bài liên quan: 242 (Valid Anagram), 383 (Ransom Note), 1657 (Determine if Two Strings Are Close), 2287 (Rearrange Characters to Make Target String), 1347.

### Biến thể 1: chuỗi chung `S` là gì?

Nếu có thể, trả về chuỗi `S` (chữ cái theo thứ tự abc) mà mọi chuỗi trở thành.

In [22]:
def common_string(words):
    n = len(words)
    total = Counter("".join(words))
    if any(v % n for v in total.values()):
        return None
    return "".join(ch * (total[ch] // n) for ch in sorted(total))

print(common_string(["abc", "aabc", "bc"]))
print(common_string(["ab", "a"]))

abc
None


### Biến thể 2: có thể chia thành **k** chuỗi bằng nhau (k tự chọn) không?

Gộp mọi ký tự rồi hỏi: có chia được thành `k` chuỗi giống nhau không? Điều kiện: `k` chia hết **mọi** tần suất ⇔ `k` chia hết `gcd` của các tần suất. `math.gcd` tính ước chung lớn nhất.

In [23]:
from math import gcd
from functools import reduce

def max_equal_parts(s):
    return reduce(gcd, Counter(s).values())

print(max_equal_parts("aabbcc"), max_equal_parts("aaaabbbb"), max_equal_parts("aab"))

2 4 1


`reduce(gcd, [a, b, c])` tính `gcd(gcd(a, b), c)` — áp dụng dồn hàm hai đối số lên cả list.

## 15. Bài tập tự luyện (TODO)

### Bài 1 — Dự đoán output

Không chạy code, cho biết `makeEqual` trả gì với: `["a","b"]`, `["aa","bb","ab"]`, `["abc","cab","bca"]`, `["aab","b"]`. Ghi vào `ex1_answers`.

In [24]:
ex1_answers = None   # TODO: list 4 giá trị bool

### Bài 2 — Điền chỗ trống: đếm bằng `dict.get`

In [25]:
def ex2_make_equal(words):
    n = len(words)
    cnt = {}
    for w in words:
        for ch in w:
            # TODO: tăng cnt[ch]
            pass
    # TODO: kiểm tra mọi giá trị chia hết cho n
    pass

# for w, e in examples: assert ex2_make_equal(w) == e

### Bài 3 — Tự viết bằng mảng 26 (không import)

In [26]:
def ex3_make_equal(words):
    # TODO
    pass

# for w, e in examples: assert ex3_make_equal(w) == e

### Bài 4 — Ransom Note (LeetCode 383)

Trả `True` nếu có thể tạo `ransom` từ các chữ của `magazine` (mỗi chữ dùng tối đa một lần).

In [27]:
def can_construct(ransom: str, magazine: str) -> bool:
    # TODO
    pass

# assert can_construct("aa", "aab") == True
# assert can_construct("aa", "ab") == False

### Bài 5 — Số chuỗi tối đa

Cho `s` và `target`. Mỗi chữ của `s` dùng tối đa một lần. Tạo được tối đa bao nhiêu bản sao `target`? (LeetCode 2287.)

In [28]:
def max_copies(s: str, target: str) -> int:
    # TODO
    pass

# assert max_copies("ilovecodingonleetcode", "code") == 2
# assert max_copies("abcba", "abc") == 1

### Bài 6 — Mini challenge: số phép toán ít nhất

Nếu có thể làm bằng nhau, trả về **số phép toán tối thiểu** (mỗi phép chuyển 1 ký tự), ngược lại `-1`. Gợi ý: tổng số ký tự "thừa" so với `S`.

In [29]:
def min_moves(words):
    # TODO
    pass

# assert min_moves(["abc", "aabc", "bc"]) == 1
# assert min_moves(["ab", "a"]) == -1
# assert min_moves(["aaa", "bbb", "ccc"]) == 6

## 16. Gợi ý 3 mức

**Bài 1.** (1) Đếm tổng mỗi chữ, chia `n`. (2) `["a","b"]`: a:1, b:1, n=2. (3) Ca thứ 4: chữ `b` xuất hiện ở cả hai chuỗi — đừng đếm sót.

**Bài 2.** (1) Mẫu đếm dict. (2) `cnt[ch] = cnt.get(ch, 0) + 1`. (3) `return all(v % n == 0 for v in cnt.values())`.

**Bài 3.** (1) `freq = [0] * 26`. (2) `freq[ord(ch) - ord('a')] += 1`. (3) Ô 0 vẫn chia hết.

**Bài 4.** (1) Đếm cả hai. (2) Với mỗi chữ của `ransom`, cần `need[c] <= have[c]`. (3) `Counter(ransom) - Counter(magazine)` rỗng ⇔ đủ.

**Bài 5.** (1) Mỗi chữ `c` của target cho phép tối đa `have[c] // need[c]` bản. (2) Lấy min trên các chữ của target. (3) `min(have[c] // need[c] for c in need)`.

**Bài 6.** (1) Mỗi phép toán giảm đúng một ký tự thừa. (2) Thừa của chuỗi `i` với chữ `c` là `max(0, cnt_i[c] - S[c])`. (3) Cộng mọi phần thừa.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [30]:
ans_ex1 = [False, True, True, True]


def ans_ex2_make_equal(words):
    n = len(words)
    cnt = {}
    for w in words:
        for ch in w:
            cnt[ch] = cnt.get(ch, 0) + 1
    return all(v % n == 0 for v in cnt.values())


def ans_ex3_make_equal(words):
    n = len(words)
    freq = [0] * 26
    for w in words:
        for ch in w:
            freq[ord(ch) - ord("a")] += 1
    return all(f % n == 0 for f in freq)


def ans_can_construct(ransom, magazine):
    return not (Counter(ransom) - Counter(magazine))


def ans_max_copies(s, target):
    have, need = Counter(s), Counter(target)
    return min(have[c] // need[c] for c in need)


def ans_min_moves(words):
    n = len(words)
    total = Counter("".join(words))
    if any(v % n for v in total.values()):
        return -1
    S = {ch: v // n for ch, v in total.items()}
    moves = 0
    for w in words:
        cw = Counter(w)
        moves += sum(max(0, cw[ch] - S[ch]) for ch in S)
    return moves


assert ans_ex1 == [make_equal_counter(w) for w in [["a", "b"], ["aa", "bb", "ab"], ["abc", "cab", "bca"], ["aab", "b"]]]
for w, e in examples:
    assert ans_ex2_make_equal(w) == e and ans_ex3_make_equal(w) == e
assert ans_can_construct("aa", "aab") and not ans_can_construct("aa", "ab")
assert ans_max_copies("ilovecodingonleetcode", "code") == 2
assert ans_max_copies("abcba", "abc") == 1
assert ans_min_moves(["abc", "aabc", "bc"]) == 1
assert ans_min_moves(["ab", "a"]) == -1
assert ans_min_moves(["aaa", "bbb", "ccc"]) == 6
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Giải thích Bài 1: `["a","b"]` → a:1 không chia hết cho 2 → `False`; `["aa","bb","ab"]` → a:3, b:3, n=3 → `True`; `["abc","cab","bca"]` → mỗi chữ 3 lần, n=3 → `True`; `["aab","b"]` → a:2, b:2 (chữ `b` có ở **cả hai** chuỗi), n=2 → `True`. Ca cuối dễ đếm sót — luôn cộng trên toàn mảng.

## 18. Template nhớ nhanh + checklist submit

```python
from collections import Counter

class Solution:
    def makeEqual(self, words):
        n = len(words)
        total = Counter("".join(words))
        return all(v % n == 0 for v in total.values())
```

**Khung tư duy:** phép toán di chuyển tự do → tìm **bất biến** (tần suất từng chữ) → điều kiện chia đều `% n == 0`.

**Checklist trước khi submit:**

- [ ] Đếm trên **toàn bộ** mảng, không theo từng chuỗi?
- [ ] Chia cho `n = len(words)` (số chuỗi), không phải số chữ khác nhau?
- [ ] Viết rõ `% n == 0`?
- [ ] Không chỉ kiểm tra tổng độ dài?
- [ ] Đã `from collections import Counter` nếu dùng Counter? Tên method `makeEqual`?